# Proyecto: Agenda de contactos interactiva

## Aplicación paso a paso de listas, tuplas, diccionarios, booleanos y control de flujo

En este proyecto construiremos una aplicación de consola que permite gestionar contactos con nombre y teléfono.

## Objetivos de aprendizaje

Al finalizar serás capaz de:

- Elegir una estructura de datos apropiada para un problema real.
- Crear funciones reutilizables.
- Usar diccionarios, listas y tuplas con una finalidad concreta.
- Tomar decisiones con valores booleanos e `if-elif-else`.
- Validar entradas de texto con `input()`.
- Depurar un programa consultando `type()`, `dir()` y `help()`.

# 1. El reto

La agenda debe permitir:

1. Añadir un contacto (nombre y teléfono).
2. Buscar un contacto por nombre.
3. Editar el teléfono de un contacto existente.
4. Eliminar un contacto.
5. Mostrar todos los contactos.

Además añadiremos la opción `0` para salir del programa de forma controlada.

## Paso 1 — Elegir las estructuras de datos

Cada contacto relaciona un **nombre** con un **teléfono**. Un diccionario es la elección natural: la clave será el nombre y el valor será el teléfono.

Usaremos también:

- Una **tupla** para las opciones del menú, ya que son fijas.
- Una **lista** para mostrar los nombres ordenados.
- Valores **booleanos** para indicar si una acción tuvo éxito.

In [ ]:
# Datos iniciales para realizar pruebas
agenda = {
    "Ana": "600 123 456",
    "Bruno": "611 987 654"
}

opciones_menu = (
    "1. Añadir un contacto",
    "2. Buscar un contacto",
    "3. Editar un contacto",
    "4. Eliminar un contacto",
    "5. Mostrar todos los contactos",
    "0. Salir"
)

print(type(agenda))
print(type(opciones_menu))
print(agenda)

## Investiga antes de programar

Necesitaremos saber si una clave está en un diccionario, añadir o modificar un par clave-valor y eliminarlo.

Ejecuta la siguiente celda. Localiza en la salida métodos como `get`, `pop`, `keys` e `items`. Consulta después la ayuda de uno de ellos.

In [ ]:
metodos_diccionario = [metodo for metodo in dir(agenda) if not metodo.startswith("_")]
print(metodos_diccionario)
help(dict.pop)

# 2. Normalizar y validar datos

Las personas pueden escribir espacios extra o nombres con mayúsculas inconsistentes. Antes de guardar o buscar, normalizaremos el nombre con `strip()` y `title()`.

Para este proyecto consideraremos válido un teléfono que contenga entre 9 y 15 dígitos, permitiendo espacios, guiones y el prefijo `+`.

In [ ]:
def normalizar_nombre(nombre):
    """Elimina espacios sobrantes y normaliza el uso de mayúsculas."""
    return nombre.strip().title()

def telefono_valido(telefono):
    """Devuelve True si el teléfono contiene entre 9 y 15 dígitos."""
    limpio = telefono.replace(" ", "").replace("-", "").replace("+", "")
    return limpio.isdigit() and 9 <= len(limpio) <= 15

for prueba in ["  aNa  ", "600 123 456", "123", "600-123-456"]:
    print(f"{prueba!r:18} -> nombre: {normalizar_nombre(prueba)!r}; teléfono válido: {telefono_valido(prueba)}")

## Pausa y piensa

- ¿Por qué `telefono_valido` devuelve un booleano?
- ¿Por qué no conviene guardar el teléfono como `int`?

Pista: un teléfono no se usa para calcular; puede incluir ceros iniciales, prefijos y separadores. Es un dato de texto (`str`).

# 3. Añadir contactos

Una buena función hace una tarea concreta. `anadir_contacto` recibe los datos y el diccionario, valida la entrada y devuelve un mensaje para informar a quien usa el programa.

In [ ]:
def anadir_contacto(agenda, nombre, telefono):
    """Añade un contacto si nombre y teléfono son válidos."""
    nombre = normalizar_nombre(nombre)

    if not nombre:
        return False, "El nombre no puede estar vacío."
    if not telefono_valido(telefono):
        return False, "El teléfono debe contener entre 9 y 15 dígitos."
    if nombre in agenda:
        return False, f"{nombre} ya existe en la agenda."

    agenda[nombre] = telefono.strip()
    return True, f"Contacto {nombre} añadido correctamente."

exito, mensaje = anadir_contacto(agenda, "  Carla  ", "622 111 222")
print(exito, mensaje)
print(agenda)

## ¿Qué devuelve la función?

La expresión `return True, mensaje` devuelve una **tupla** con dos valores. Así la función comunica dos cosas: si tuvo éxito y un texto explicativo.

El desempaquetado `exito, mensaje = ...` guarda cada componente de la tupla en una variable distinta.

# 4. Buscar un contacto

Para buscar, primero normalizamos el nombre. El método `.get()` permite obtener el teléfono sin provocar un error cuando no existe la clave.

In [ ]:
def buscar_contacto(agenda, nombre):
    """Busca un contacto y devuelve su teléfono o None."""
    nombre = normalizar_nombre(nombre)
    return agenda.get(nombre)

for nombre in ["ana", "Carla", "Daniel"]:
    telefono = buscar_contacto(agenda, nombre)
    if telefono is not None:
        print(f"{normalizar_nombre(nombre)}: {telefono}")
    else:
        print(f"No se encontró a {normalizar_nombre(nombre)}.")

# 5. Editar un contacto

Editar requiere dos comprobaciones: el contacto debe existir y el nuevo teléfono debe ser válido. `if-elif-else` expresa claramente estas alternativas.

In [ ]:
def editar_contacto(agenda, nombre, nuevo_telefono):
    """Actualiza el teléfono de un contacto existente."""
    nombre = normalizar_nombre(nombre)

    if nombre not in agenda:
        return False, f"No existe ningún contacto llamado {nombre}."
    elif not telefono_valido(nuevo_telefono):
        return False, "El teléfono no tiene un formato válido."
    else:
        agenda[nombre] = nuevo_telefono.strip()
        return True, f"Teléfono de {nombre} actualizado."

exito, mensaje = editar_contacto(agenda, "Carla", "633 333 333")
print(exito, mensaje)
print(buscar_contacto(agenda, "carla"))

# 6. Eliminar un contacto

El método `.pop(clave)` elimina una entrada y devuelve el valor que había guardado. Primero comprobamos que el contacto existe para evitar un `KeyError`.

In [ ]:
def eliminar_contacto(agenda, nombre):
    """Elimina un contacto existente de la agenda."""
    nombre = normalizar_nombre(nombre)

    if nombre not in agenda:
        return False, f"No existe ningún contacto llamado {nombre}."

    telefono = agenda.pop(nombre)
    return True, f"Contacto {nombre} ({telefono}) eliminado."

exito, mensaje = eliminar_contacto(agenda, "Bruno")
print(exito, mensaje)
print(agenda)

# 7. Mostrar todos los contactos

Una agenda clara debería mostrar sus contactos en orden alfabético. `sorted(agenda)` devuelve una **lista** de las claves ordenadas sin modificar el diccionario.

In [ ]:
def mostrar_contactos(agenda):
    """Muestra todos los contactos en orden alfabético."""
    if not agenda:
        print("La agenda está vacía.")
        return

    print(f"{'NOMBRE':<20} TELÉFONO")
    print("-" * 35)
    for nombre in sorted(agenda):
        print(f"{nombre:<20} {agenda[nombre]}")

mostrar_contactos(agenda)
print(type(sorted(agenda)))

# 8. Integrar el menú

Ya tenemos las piezas. Ahora construiremos una función que muestra el menú en un bucle `while`, recibe una opción y llama a la función correspondiente.

La variable `en_ejecucion` es booleana: controla cuándo debe continuar el programa.

In [ ]:
def mostrar_menu():
    print("\nAGENDA DE CONTACTOS")
    print("-" * 22)
    for opcion in opciones_menu:
        print(opcion)

def ejecutar_agenda(agenda):
    """Ejecuta el menú interactivo hasta que la persona elige salir."""
    en_ejecucion = True

    while en_ejecucion:
        mostrar_menu()
        opcion = input("Elige una opción: " ).strip()

        if opcion == "1":
            nombre = input("Nombre: " )
            telefono = input("Teléfono: " )
            _, mensaje = anadir_contacto(agenda, nombre, telefono)
            print(mensaje)
        elif opcion == "2":
            nombre = input("Nombre que deseas buscar: " )
            telefono = buscar_contacto(agenda, nombre)
            if telefono is None:
                print("Contacto no encontrado.")
            else:
                print(f"{normalizar_nombre(nombre)}: {telefono}")
        elif opcion == "3":
            nombre = input("Contacto que deseas editar: " )
            telefono = input("Nuevo teléfono: " )
            _, mensaje = editar_contacto(agenda, nombre, telefono)
            print(mensaje)
        elif opcion == "4":
            nombre = input("Contacto que deseas eliminar: " )
            _, mensaje = eliminar_contacto(agenda, nombre)
            print(mensaje)
        elif opcion == "5":
            mostrar_contactos(agenda)
        elif opcion == "0":
            en_ejecucion = False
            print("Hasta pronto.")
        else:
            print("Opción no válida. Elige un número del menú.")

## Ejecutar la aplicación

Ejecuta primero todas las celdas anteriores, en orden. Después descomenta la siguiente línea para iniciar la agenda.

Prueba a añadir, buscar, editar, eliminar y mostrar contactos. Usa `0` cuando quieras terminar.

In [ ]:
# ejecutar_agenda(agenda)

# 9. Comprueba lo aprendido

Responde y prueba en el código:

1. ¿Qué estructura contiene la agenda y qué representa cada clave?
2. ¿Por qué `opciones_menu` es una tupla?
3. ¿Qué devuelve `anadir_contacto` y cómo se desempaqueta?
4. ¿Qué condición evita un `KeyError` al eliminar?
5. ¿Dónde se usa una lista en el proyecto?
6. ¿Qué booleano controla el bucle principal?

## Ejercicios de ampliación

### Nivel 1
- Impide añadir un contacto si el número de teléfono ya está asignado a otra persona.
- Añade una opción para contar los contactos de la agenda.
- Muestra los contactos cuyo nombre comienza por una letra elegida.

### Nivel 2
- Permite almacenar también correo electrónico y valida que contenga `@`.
- Pide confirmación antes de eliminar un contacto.
- Implementa la búsqueda parcial: que `"an"` encuentre `"Ana"`.

### Nivel 3
- Guarda la agenda en un archivo de texto y recupérala al iniciar el programa.
- Añade grupos de contactos, como familia, clase o trabajo.

# Resumen

Has creado una aplicación completa a partir de pequeñas funciones.

- Los **diccionarios** guardan la relación nombre-teléfono.
- Las **tuplas** agrupan valores fijos y resultados de funciones.
- Las **listas** permiten trabajar con los nombres ordenados.
- Los **booleanos** y `if-elif-else` deciden qué acción realizar.
- `while` mantiene la agenda activa hasta que se elige salir.

El siguiente paso natural es persistir los contactos en un archivo.